In [2]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any

In [3]:
load_dotenv("E:\\Workspace\\Agentic Job Assistant\\ai-job-hunter\\.env")

True

In [4]:
model = ChatOpenRouter(model="anthropic/claude-sonnet-4.6", temperature=0, app_title=None)

In [5]:
model.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'model_name': 'anthropic/claude-4.6-sonnet-20260217', 'id': 'gen-1774768724-s5AJ7gz8M5AdWjwUMpwP', 'created': 1774768724, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019d3875-9a6b-7bd0-9339-dacd2af7d99e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 11, 'total_tokens': 25, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

In [6]:
RESUME_PROMPT = """
# 🛠️ LaTeX Resume Architect & Alignment Strategist

## 🎯 Core Objective
You are an expert Resume Strategist and LaTeX Developer. Your goal is to optimize a candidate's resume to strictly align with a provided Job Description (JD) to pass both ATS software and human review.

**Primary Directives:**
1.  **Prioritize Outcome-Focused Content:** Use the XYZ Formula (Action + Context + Result). Start EVERY bullet with a strong, distinct action verb.
2.  **Preserve Excellence:** If an existing bullet point is already strong, metric-heavy, and JD-aligned, **do not rewrite it.** Output it exactly as is.
3.  **Mine Hidden Content:** Analyze **commented-out LaTeX code** (lines starting with `%`). If a commented project or experience aligns better with the JD than an active one, **swap them.**
4.  **Remove Fluff:** Eliminate "Responsible for," "Assisted," "Worked on," or generic duties.
5.  **ATS Safety Protocol:** Use only simple, standard LaTeX environments (like standard `itemize`). Avoid complex macros, multi-columns, or `minipage` to ensure clean PDF-to-text parsing.

The Output must be modular, section-by-section, and provided in raw LaTeX code.

---
## 1. The Strategy (The "XYZ" Formula)
For every *new* or *rewritten* bullet point, apply the Google XYZ formula:
* "Accomplished [X] as measured by [Y], by doing [Z]."
* *Constraint:* Every bullet point must contain a metric or a specific technical outcome.

**The "Good Enough" Clause:**
Before rewriting a bullet, ask: *Does this already follow XYZ and address a specific JD requirement?*
* **YES:** Keep it verbatim. Note this in the rationale.
* **NO:** Rewrite it.

---
## 2. Execution Process & Output Generation
Analyze the Provided Resume and Target Job Description. Generate the following distinct sections:

### Section 0: The Hidden Content Audit
* **Goal:** A brief text summary of what you are swapping.
* **Output:** Simply state if you found any commented-out (`%`) projects/bullets that better match the JD and confirm what active content they will replace. If none, state "No swaps needed."

### Section 1: Executive Highlights (The Hook)
* **Goal:** Create a high-impact, 3-bullet summary that immediately qualifies the candidate. Max 35 words per bullet.
* **Structure:**
    1.  **Foundation:** Education + Total Years of Experience + The JD's "Must-Have" technical domain.
    2.  **Bridge:** Soft Skills/Collaboration + Business Alignment.
    3.  **Specialist:** Advanced/Niche Tech + A specific problem-solving outcome.
* **LaTeX Output:** Provide the code inside an `itemize` environment. Use `\\textbf{{}}` to highlight the most critical skill.

### Section 2: Skills (The "ATS Shield")
* **Goal:** Reorder and rename candidate skills to match the JD's exact hierarchy.
* **Constraint:** You MUST use the **exact spelling and casing** found in the JD (e.g., if the JD says "Node.js", do not write "NodeJS").
* **LaTeX Output:** Provide the updated skills section code.

### Section 3: Experience & Projects (The "Meat")
* **Goal:** Output the final LaTeX for the roles and projects.
* **The Change:**
    * Weak bullets: Rewrite to XYZ format focusing on JD problems.
    * Strong bullets: Keep as is.
    * Swapped bullets: Uncomment and polish the hidden content identified in Section 0.
* **LaTeX Output:** Provide the `\\item` lines.
* **Rationale:** Below the LaTeX block for each role/project, briefly explain if bullets were **Preserved**, **Rewritten**, or **Swapped from Comments** and why.

---
## 3. Formatting Rules (LaTeX)
* Use standard LaTeX syntax.
* **Escape special characters:** Use `\\%` for percentages, `\\$` for dollars, `\\&` for ampersands.
* Do not include the preamble (`\\documentclass`), only the inner content.
* Use `\\textbf{{}}` to bold key metrics/tech.

---
"""

In [7]:
with open("E:\\Workspace\\Agentic Job Assistant\\ai-job-hunter\\templates\\resume_templates\\data_science_resume.tex", "r", encoding="utf-8") as f:
    resume_tex = f.readlines()

with open("E:\\Workspace\\Agentic Job Assistant\\ai-job-hunter\\examples\\jerry_data_scientist.txt", "r", encoding="utf-8") as f:
    job_description = f.readlines()

In [8]:
prompt = [
    SystemMessage(content=RESUME_PROMPT),
    HumanMessage(content="Here is the candidate's current resume in LaTeX format:\n" + "".join(resume_tex)),
    HumanMessage(content="Here is the target Job Description:\n" + "".join(job_description)),
    HumanMessage(content="Please provide the optimized resume sections in LaTeX code, along with your rationale for each change.")
]
response = model.invoke(prompt)

In [ ]:
class StrategicNotes(BaseModel):
    summary: str = Field(description="High-level strategic takeaway for this section.")
    priorities: List[str] = Field(default_factory=list, description="What matters most in this section.")
    risks: List[str] = Field(default_factory=list, description="Potential weaknesses or concerns.")
    opportunities: List[str] = Field(default_factory=list, description="Ways to improve alignment.")

class LatexRewriteBlock(BaseModel):
    latex: str = Field(description="The rewritten LaTeX code for this section.")
    notes: Optional[str] = None

class RationaleRow(BaseModel):
    target: str = Field(description="The specific bullet point or skill being evaluated.")
    action: str = Field(description="Whether it was Preserved, Rewritten, or Swapped from Comments.")
    reason: str = Field(description="The rationale behind the action taken.")

class HighlightsSection(BaseModel):
    strategy: StrategicNotes
    rewritten_latex: LatexRewriteBlock
    rationale: List[RationaleRow]

class SkillsSection(BaseModel):
    strategy: StrategicNotes
    rewritten_latex: LatexRewriteBlock
    rationale: List[RationaleRow]

class ExperienceSection(BaseModel):
    section_id: str
    title: str
    strategy: StrategicNotes
    jd_alignment_check: List[str]
    rewritten_latex: LatexRewriteBlock
    rationale: List[RationaleRow]

In [10]:
print(response.content)

# Resume Optimization Analysis for Jerry.ai — Junior Applied Data Scientist

---

## Section 0: The Hidden Content Audit

**Swap Analysis:**

After reviewing all commented-out (`%`) blocks against the JD, here is the verdict:

- **`ClotSense – Medical Imaging Classification System`** (commented): Contains strong data pipeline, validation, and benchmarking language — but is CV/medical imaging focused, which is tangential to Jerry.ai's auto-insurance/fintech domain. **No swap.**
- **`GenZ AI Therapist`** (commented): Multi-agent LLM project — interesting but not aligned with data quality, feature stores, or statistical modeling priorities. **No swap.**
- **`AI Delivery Tracking Voice Agent`** (commented): FastAPI + schema validation + hallucination-safe response builder — the schema validation and deterministic tool contracts *do* align with the JD's "data accuracy" and "automated frameworks" themes. However, the active projects (Churn, NYC 311) are stronger overall fits. **No swap.**
- 